# 03 - Prepare & Export
Package the three published datasets - one per social chart - as CSV plus a
plain-English codebook for every column (fun-tier release = CSV + codebook only):
1. `highest_grossing_films_v1` - top domestic films, ticket-price adjusted - chart 2.
2. `films_worldwide_v1` - worldwide grosses with the home-vs-abroad split - chart 1.
3. `films_foreign_us_v1` - top foreign-LANGUAGE films by U.S. gross (nominal) - chart 3.

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import pandas as pd
from src.ingest import load_config
from src.clean_quality import get_connection
from src.prepare import package_dataset
cfg = load_config('config.yaml')
con = get_connection(cfg)
# The three published datasets match the three social charts. Fun tier =
# CSV + codebook only (formats=['csv']); the richer Excel/Parquet stay local.
films = con.execute('SELECT * FROM films_adjusted ORDER BY adjusted_gross DESC').df()
worldwide = con.execute('SELECT * FROM films_worldwide ORDER BY worldwide_gross DESC').df()
foreign = con.execute('SELECT * FROM films_foreign_us ORDER BY domestic_gross DESC').df()
print('adjusted', films.shape, '| worldwide', worldwide.shape, '| foreign', foreign.shape)

## Codebook — a plain-English description for every column

In [ ]:
# Chart 2 dataset: top domestic films, adjusted for ticket-price inflation (BOM).
codebook = {
    'rank_adjusted':      'Rank by ticket-price-adjusted domestic gross (1 = highest).',
    'title':              'Film title.',
    'adjusted_gross':     "Domestic (U.S. & Canada) lifetime gross, adjusted for ticket-price inflation via Box Office Mojo (estimated tickets x today's average ticket price) - an admissions basis (USD).",
    'nominal_gross':      'Domestic lifetime gross in year-of-release dollars, as originally reported (USD).',
    'est_tickets':        'Estimated number of tickets sold over the film lifetime (Box Office Mojo estimate).',
    'release_year':       'Year of the film original theatrical release.',
    'decade':             'Release decade (release_year rounded down to the nearest 10).',
    'inflation_multiple': 'adjusted_gross / nominal_gross - how many times its original take the adjusted figure represents.',
}
notes = '''
Source: Box Office Mojo, Top Lifetime Adjusted Grosses (domestic, US/Canada).
URL: https://www.boxofficemojo.com/chart/top_lifetime_gross_adjusted/?adjust_gross_to=2022
Method: adjusted_gross is Box Office Mojo's ticket-price adjustment - estimated tickets sold x a
reference-year average ticket price, i.e. an ADMISSIONS basis (counts people through the door), the
sound way to compare films across eras. NOT a CPI/general-inflation recalculation (a CPI version was
tried and reverted because it overstates old films unevenly by era). Domestic only; lifetime totals
include re-release grosses, which inflates some classics.
'''
written = package_dataset(films, cfg, name='highest_grossing_films_v1', codebook=codebook, notes=notes, formats=['csv'])
written

## Package the worldwide + foreign-language exports

In [ ]:
# Chart 1 dataset: worldwide grosses with the home-vs-abroad split.
ww_codebook = {
    'rank_worldwide': 'Rank by worldwide lifetime gross (1 = highest).',
    'title': 'Film title.',
    'worldwide_gross': 'Worldwide lifetime gross, nominal USD (domestic + foreign).',
    'domestic_gross': 'Domestic (U.S. & Canada) lifetime gross, nominal USD.',
    'foreign_gross': 'International (rest-of-world) lifetime gross, nominal USD.',
    'release_year': 'Year of original theatrical release.',
    'domestic_pct': 'Domestic gross as a percent of worldwide.',
    'foreign_pct': 'International gross as a percent of worldwide.',
}
ww_notes = '''Source: Box Office Mojo, Top Lifetime Grosses (Worldwide).
Nominal (year-of-release) dollars, NOT inflation-adjusted. Domestic = US & Canada.'''
package_dataset(worldwide, cfg, name='films_worldwide_v1', codebook=ww_codebook, notes=ww_notes, formats=['csv'])

# Chart 3 dataset: top foreign-LANGUAGE films by U.S. box office (nominal $).
foreign_codebook = {
    'rank_foreign': 'Rank among foreign-language films by U.S. & Canada gross (1 = highest).',
    'title': 'Film title.',
    'domestic_gross': 'U.S. & Canada lifetime gross in NOMINAL (year-of-release) dollars, as reported (USD).',
    'release_year': 'Year of original theatrical release.',
    'distributor': 'U.S. distributor as listed by Box Office Mojo.',
    'origin_country': 'Country-of-origin code (TMDB).',
    'origin_name': 'Country-of-origin name (TMDB), shown under each title on the chart.',
}
foreign_notes = '''Source: Box Office Mojo, Foreign Language chart (non-English-language films),
ranked by U.S. & Canada lifetime gross; country of origin joined from TMDB. Grosses are NOMINAL
(year-of-release dollars) - no reliable admissions/ticket-price adjustment exists for this list, so
older titles are modestly understated. "Foreign" here means LANGUAGE (primary language not English).
"This product uses the TMDB API but is not endorsed or certified by TMDB."'''
package_dataset(foreign, cfg, name='films_foreign_us_v1', codebook=foreign_codebook, notes=foreign_notes, formats=['csv'])

---
**Next:** `04-viz.ipynb` (exploration) and `06-viz-social.ipynb` (the three social charts).

## Cleanup

In [ ]:
con.close()
print('connection closed')